This is our multi step algorithm to determine best locations for a coffee shop given our knowledge graph

1. Build a Node Regression Pipeline: 
    - Target Output: Avg_rating of a BusinessLocation Node
    - Input Features: Block Group attributes, Business Location Node locations, BusLoc - BusLoc relationships)

2. Extract Weights from Node Regression

3. Generate an "Optimal" Block Group

4. Similarity Search to Find Real Block Groups Closest to the Optimal Block Group
    - Rank by closeness in vector space 

5. Generate 10 New Sample Locations to Test
    - Use the geographical distribution of the businesses and county boundaries 
    - Test against zone locations to auto disqualify

6. Input the Generated Sample Locations into the Knowledge Graph

7. Use Node Regression Pipeline to Return Avg_Rating
    - Rank locations based on rating and other criteria


### 1. Setup

In [1]:
import pandas as pd

# import geopandas as gpd

import os

from dotenv import load_dotenv
from decimal import Decimal
from neo4j import GraphDatabase


In [2]:



group_driver = GraphDatabase.driver(
     "bolt://67.58.49.87:7687",
     auth=("neo4j", "h2u9l4px")
)


with group_driver.session() as session:
    result = session.run("MATCH (n) UNWIND labels(n) AS label RETURN count(DISTINCT label) AS count")
    num_nodes = result.single()["count"]
    print(f"Connection Successful: {num_nodes} unique node types found in the graph database")


Connection Successful: 11 unique node types found in the graph database


### 2. Best Location Algorithm

#### 2.1 Node Regression

##### 2.1.1 Configure Pipeline

In [33]:
pipeline_query = """ 
CALL gds.alpha.pipeline.nodeRegression.create('pipe')

YIELD name, nodePropertySteps, featureProperties, splitConfig, autoTuningConfig, parameterSpace
"""

with group_driver.session() as session:
    result = session.run(pipeline_query)
    for record in result:
        print(record)




ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.alpha.pipeline.nodeRegression.create`: Caused by: java.lang.IllegalStateException: Pipeline named `pipe` already exists.} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}

In [34]:
# feature_columns = [ 'zip', 'latitude', 'longitude', 'blockgroup', 'franchise_bool','medhinc_cy', 'avghinc_cy', 'gini_fy',
#        'indmanu_cy', 'totpop_cy', 'fem25', 'fem30', 'fem35', 'male25',
#        'male30', 'male35', 'crmcytotc', 'di100_cy', 'di150_cy']


add_property_query = """CALL gds.alpha.pipeline.nodeRegression.addNodeProperty('pipe', 'scaleProperties', {
  nodeProperties: 'avg_rating',
  scaler: 'MinMax',
  mutateProperty:'scaled_rating'
}) YIELD name, nodePropertySteps"""

with group_driver.session() as session:
    result = session.run(add_property_query)
    for record in result:
        print(record)


ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.alpha.pipeline.nodeRegression.addNodeProperty`: Caused by: java.lang.IllegalArgumentException: The value of `mutateProperty` is expected to be unique, but scaled_rating was already specified in the gds.scaleProperties.mutate procedure.} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}

In [35]:
select_feature_query = """
CALL gds.alpha.pipeline.nodeRegression.selectFeatures('pipe', ['scaled_rating', 'avg_rating'])
YIELD name, featureProperties"""

with group_driver.session() as session:
    result = session.run(select_feature_query)
    for record in result:
        print(record)


<Record name='pipe' featureProperties=['scaled_rating', 'avg_rating', 'scaled_rating', 'avg_rating']>


In [36]:
split_query = """
CALL gds.alpha.pipeline.nodeRegression.configureSplit('pipe', {
  testFraction: 0.2,
  validationFolds: 5
}) YIELD splitConfig"""


with group_driver.session() as session:
    result = session.run(split_query)
    for record in result:
        print(record)


<Record splitConfig={'testFraction': 0.2, 'validationFolds': 5}>


In [31]:
add_model_query = """

CALL gds.alpha.pipeline.nodeRegression.addLinearRegression('pipe')
YIELD parameterSpace"""

with group_driver.session() as session:
    result = session.run(add_model_query)
    for record in result:
        print(record)


<Record parameterSpace={'LinearRegression': [{'maxEpochs': 100, 'minEpochs': 1, 'penalty': 0.0, 'patience': 1, 'methodName': 'LinearRegression', 'batchSize': 100, 'tolerance': 0.001, 'learningRate': 0.001}, {'maxEpochs': 100, 'minEpochs': 1, 'penalty': 0.0, 'patience': 1, 'methodName': 'LinearRegression', 'batchSize': 100, 'tolerance': 0.001, 'learningRate': 0.001}], 'RandomForest': []}>


##### 2.1.2 Train Pipeline

In [32]:
project_graph_query = """
MATCH (bl:BusinessLocation)
RETURN gds.graph.project(
  'myGraph',
  bl,
  null,
  {
    sourceNodeLabels: labels(bl),
    targetNodeLabels: [],
    sourceNodeProperties: bl { .avg_rating, .latitude, .longitude },
    targetNodeProperties: {}
  }
)"""

with group_driver.session() as session:
    result = session.run(project_graph_query)
    for record in result:
        print(record)


ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke function `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Graph myGraph already exists} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}

In [29]:
train_model_query = """

CALL gds.alpha.pipeline.nodeRegression.train('myGraph', {
  pipeline: 'pipe',
  targetNodeLabels: ['BusinessLocation'],
  modelName: 'nr-pipeline-model',
  targetProperty: 'avg_rating',
  randomSeed: 25,
  concurrency: 1,
  metrics: ['MEAN_SQUARED_ERROR']
}) YIELD modelInfo
RETURN
  modelInfo.bestParameters AS winningModel,
  modelInfo.metrics.MEAN_SQUARED_ERROR.train.avg AS avgTrainScore,
  modelInfo.metrics.MEAN_SQUARED_ERROR.outerTrain AS outerTrainScore,
  modelInfo.metrics.MEAN_SQUARED_ERROR.test AS testScore"""


with group_driver.session() as session:
    result = session.run(train_model_query)
    for record in result:
        print(record)


ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.alpha.pipeline.nodeRegression.train`: Caused by: java.lang.IllegalArgumentException: Node with id 53493 has `avg_rating` target property value `NaN`} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}

In [28]:

add_context_query = """

MATCH (bl:BusinessLocation)
OPTIONAL MATCH (bl:BusinessLocation)-[r:contained_in]->(bg:BlockGroup)
RETURN gds.graph.project(
  'bus_in_blockgroup_Graph',
  bl,
  bg,
  {
    sourceNodeLabels: ['bg'],
    targetNodeLabels: ['bl'],
    sourceNodeProperties: bg {avghinc_cy, totpop_cy, crmcytotc},
    targetNodeProperties: bl {avg_rating, latitude, longitude },
    relationshipType: 'contained_in'
  },
  { undirectedRelationshipTypes: ['contained_in'] }
)"""

with group_driver.session() as session:
    result = session.run(add_context_query)
    for record in result:
        print(record)



ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project.cypher`: Caused by: java.lang.IllegalArgumentException: Invalid key: nodeProperties} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}

In [17]:
create_pipe_w_context_query = """
CALL gds.alpha.pipeline.nodeRegression.create('pipe-with-context')"""

with group_driver.session() as session:
    result = session.run(create_pipe_w_context_query)
    print(result)



In [18]:
add_node_query = """
CALL gds.alpha.pipeline.nodeRegression.addNodeProperty('pipe-with-context', 'fastRP', {
  embeddingDimension: 64,
  iterationWeights: [0, 1],
  mutateProperty:'embedding',
  contextNodeLabels: ['BlockGroup'],
  randomSeed: 1337
})"""

with group_driver.session() as session:
    result = session.run(add_node_query)
    print(result)


In [19]:
add_embedding_query = """ 
CALL gds.alpha.pipeline.nodeRegression.selectFeatures('pipe-with-context', ['embedding'])"""

with group_driver.session() as session:
    result = session.run(add_embedding_query)
    print(result)




In [23]:
add_model_query = """ 
CALL gds.alpha.pipeline.nodeRegression.addRandomForest('pipe-with-context', {numberOfDecisionTrees: 5})"""

with group_driver.session() as session:
    result = session.run(add_model_query)
    for record in result:
        print(record)




<Record name='pipe-with-context' nodePropertySteps=[{'name': 'gds.fastRP.mutate', 'config': {'randomSeed': 1337, 'contextRelationshipTypes': [], 'iterationWeights': [0, 1], 'embeddingDimension': 64, 'contextNodeLabels': ['BlockGroup'], 'mutateProperty': 'embedding'}}] featureProperties=['embedding'] splitConfig={'testFraction': 0.3, 'validationFolds': 3} autoTuningConfig={'maxTrials': 10} parameterSpace={'LinearRegression': [], 'RandomForest': [{'maxDepth': 2147483647, 'minLeafSize': 1, 'minSplitSize': 2, 'numberOfDecisionTrees': 5, 'methodName': 'RandomForest', 'numberOfSamplesRatio': 1.0}, {'maxDepth': 2147483647, 'minLeafSize': 1, 'minSplitSize': 2, 'numberOfDecisionTrees': 5, 'methodName': 'RandomForest', 'numberOfSamplesRatio': 1.0}, {'maxDepth': 2147483647, 'minLeafSize': 1, 'minSplitSize': 2, 'numberOfDecisionTrees': 5, 'methodName': 'RandomForest', 'numberOfSamplesRatio': 1.0}]}>


In [24]:
  

train_model_query = """ 
CALL gds.alpha.pipeline.nodeRegression.train('bus_in_blockgroup_Graph', {
  pipeline: 'pipe-with-context',
  targetNodeLabels: ['BusinessLocation'],
  modelName: 'nr-pipeline-model-contextual',
  targetProperty: 'avg_rating',
  randomSeed: 25,
  concurrency: 1,
  metrics: ['MEAN_SQUARED_ERROR']
}) YIELD modelInfo
RETURN
  modelInfo.bestParameters AS winningModel,
  modelInfo.metrics.MEAN_SQUARED_ERROR.train.avg AS avgTrainScore,
  modelInfo.metrics.MEAN_SQUARED_ERROR.outerTrain AS outerTrainScore,
  modelInfo.metrics.MEAN_SQUARED_ERROR.test AS testScore
"""

with group_driver.session() as session:
    result = session.run(train_model_query)
    for record in result:
        print(record)




ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.alpha.pipeline.nodeRegression.train`: Caused by: org.neo4j.gds.core.loading.GraphNotFoundException: Graph with name `bus_in_blockgroup_Graph` does not exist on database `neo4j`. It might exist on another database.} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}

#### 2.5 Sample Business Location Generation